# 1. Facies Augmentation

In [ ]:
# /home/g202103050/Documents/A.Rahman/2-3D GAN part/2D/2D_New Eara/augment_facies_pal_only_nn.py
from __future__ import annotations

import os
import math
import random
from typing import List, Tuple

import numpy as np
from PIL import Image, ImageOps

# ----------------------------- Paths -----------------------------
original_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_facies'
augmented_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Facies_augmented'
os.makedirs(augmented_dir, exist_ok=True)

# ------------------------- Augment Config -------------------------
AUGS_PER_IMAGE = 30
SEED = 2025

ROTATION_DEG = 10          # ±deg
SHIFT_FRAC   = 0.10        # fraction of width/height
SHEAR_X_MAX  = 0.10        # tan(theta) along x
SHEAR_Y_MAX  = 0.10        # tan(theta) along y
ZOOM_FRAC    = 0.10        # ±10%
H_FLIP       = True

# Optional upscale (leave None to keep original px size)
TARGET_LONG_SIDE_PX = None
MIN_SCALE = 1

# ----------------------------- Helpers ----------------------------

def compute_scale(w: int, h: int) -> int:
    need = (TARGET_LONG_SIDE_PX / max(w, h)) if TARGET_LONG_SIDE_PX else 1.0
    return max(MIN_SCALE, int(math.ceil(need)))

def guess_background_index(imP: Image.Image) -> int:
    """Use the modal value on a 2px border as background (keeps 'no data' color)."""
    idx = np.array(imP, dtype=np.uint8)
    border = np.concatenate([idx[:2, :].ravel(), idx[-2:, :].ravel(),
                             idx[:, :2].ravel(), idx[:, -2:].ravel()])
    vals, counts = np.unique(border, return_counts=True)
    return int(vals[np.argmax(counts)])

def ensure_palette(imgP: Image.Image, palette: List[int]) -> None:
    """Why: some ops can drop/alter palette; force the original back."""
    if palette:
        imgP.putpalette(palette)

def center_zoom_P(imgP: Image.Image, scale: float, bg_idx: int, palette: List[int]) -> Image.Image:
    w, h = imgP.size
    nw, nh = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    res = imgP.resize((nw, nh), resample=Image.NEAREST)
    ensure_palette(res, palette)
    if scale >= 1.0:
        left, top = (nw - w) // 2, (nh - h) // 2
        out = res.crop((left, top, left + w, top + h))
        ensure_palette(out, palette)
        return out
    bg = Image.new("P", (w, h), color=bg_idx)
    ensure_palette(bg, palette)
    bg.paste(res, ((w - nw) // 2, (h - nh) // 2))
    return bg

def shear_P(imgP: Image.Image, shx: float, shy: float, bg_idx: int, palette: List[int]) -> Image.Image:
    # PIL uses output->input mapping
    a, b, c = 1.0, -shx, 0.0
    d, e, f = -shy, 1.0, 0.0
    out = imgP.transform(imgP.size, Image.AFFINE, (a, b, c, d, e, f),
                         resample=Image.NEAREST, fillcolor=bg_idx)
    ensure_palette(out, palette)
    return out

def translate_P(imgP: Image.Image, dx: int, dy: int, bg_idx: int, palette: List[int]) -> Image.Image:
    a, b, c = 1.0, 0.0, -dx
    d, e, f = 0.0, 1.0, -dy
    out = imgP.transform(imgP.size, Image.AFFINE, (a, b, c, d, e, f),
                         resample=Image.NEAREST, fillcolor=bg_idx)
    ensure_palette(out, palette)
    return out

def random_augment_P(imgP: Image.Image, rng: random.Random) -> Image.Image:
    palette = imgP.getpalette()
    bg_idx = guess_background_index(imgP)
    im = imgP.copy()

    if H_FLIP and rng.random() < 0.5:
        im = ImageOps.mirror(im)
        ensure_palette(im, palette)

    z = rng.uniform(1.0 - ZOOM_FRAC, 1.0 + ZOOM_FRAC)
    im = center_zoom_P(im, z, bg_idx, palette)

    shx = rng.uniform(-SHEAR_X_MAX, SHEAR_X_MAX)
    shy = rng.uniform(-SHEAR_Y_MAX, SHEAR_Y_MAX)
    im = shear_P(im, shx, shy, bg_idx, palette)

    angle = rng.uniform(-ROTATION_DEG, ROTATION_DEG)
    im = im.rotate(angle, resample=Image.NEAREST, expand=False, fillcolor=bg_idx)
    ensure_palette(im, palette)

    w, h = im.size
    dx = int(round(rng.uniform(-SHIFT_FRAC, SHIFT_FRAC) * w))
    dy = int(round(rng.uniform(-SHIFT_FRAC, SHIFT_FRAC) * h))
    im = translate_P(im, dx, dy, bg_idx, palette)

    return im

def upscale_P(imgP: Image.Image) -> Image.Image:
    scale = compute_scale(imgP.width, imgP.height)
    if scale > 1:
        palette = imgP.getpalette()
        imgP = imgP.resize((imgP.width * scale, imgP.height * scale), resample=Image.NEAREST)
        ensure_palette(imgP, palette)
    return imgP

# ------------------------------ Main ------------------------------

def main() -> None:
    rng = random.Random(SEED)
    files = [f for f in sorted(os.listdir(original_dir)) if f.lower().endswith(('.png', '.gif'))]
    if not files:
        print(f"No images found in {original_dir}")
        return

    for i, fname in enumerate(files):
        src = os.path.join(original_dir, fname)
        base = os.path.splitext(fname)[0]

        # Keep palette; if input isn't 'P', quantize once to a palette
        im = Image.open(src)
        if im.mode != "P":
            im = im.convert("P")

        for j in range(1, AUGS_PER_IMAGE + 1):
            aug = random_augment_P(im, rng)
            aug = upscale_P(aug)  # no-op unless TARGET_LONG_SIDE_PX set
            out_path = os.path.join(augmented_dir, f"{base}_aug{j:02d}.png")
            aug.save(out_path, format="PNG", optimize=True, compress_level=9)
            print(f"Saved augmented facies image: {out_path} ({aug.width}x{aug.height}px)")

if __name__ == "__main__":
    main()


# 2. Porosity Augmentation

In [ ]:
# /home/g202103050/Documents/A.Rahman/2-3D GAN part/2D/2D_New Eara/augment_facies_pal_only_nn.py
from __future__ import annotations

import os
import math
import random
from typing import List, Tuple

import numpy as np
from PIL import Image, ImageOps

# ----------------------------- Paths -----------------------------
original_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_Porosity'
augmented_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Porosity_augmented'
os.makedirs(augmented_dir, exist_ok=True)

# ------------------------- Augment Config -------------------------
AUGS_PER_IMAGE = 30
SEED = 2025

ROTATION_DEG = 10          # ±deg
SHIFT_FRAC   = 0.10        # fraction of width/height
SHEAR_X_MAX  = 0.10        # tan(theta) along x
SHEAR_Y_MAX  = 0.10        # tan(theta) along y
ZOOM_FRAC    = 0.10        # ±10%
H_FLIP       = True

# Optional upscale (leave None to keep original px size)
TARGET_LONG_SIDE_PX = None
MIN_SCALE = 1

# ----------------------------- Helpers ----------------------------

def compute_scale(w: int, h: int) -> int:
    need = (TARGET_LONG_SIDE_PX / max(w, h)) if TARGET_LONG_SIDE_PX else 1.0
    return max(MIN_SCALE, int(math.ceil(need)))

def guess_background_index(imP: Image.Image) -> int:
    """Use the modal value on a 2px border as background (keeps 'no data' color)."""
    idx = np.array(imP, dtype=np.uint8)
    border = np.concatenate([idx[:2, :].ravel(), idx[-2:, :].ravel(),
                             idx[:, :2].ravel(), idx[:, -2:].ravel()])
    vals, counts = np.unique(border, return_counts=True)
    return int(vals[np.argmax(counts)])

def ensure_palette(imgP: Image.Image, palette: List[int]) -> None:
    """Why: some ops can drop/alter palette; force the original back."""
    if palette:
        imgP.putpalette(palette)

def center_zoom_P(imgP: Image.Image, scale: float, bg_idx: int, palette: List[int]) -> Image.Image:
    w, h = imgP.size
    nw, nh = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    res = imgP.resize((nw, nh), resample=Image.NEAREST)
    ensure_palette(res, palette)
    if scale >= 1.0:
        left, top = (nw - w) // 2, (nh - h) // 2
        out = res.crop((left, top, left + w, top + h))
        ensure_palette(out, palette)
        return out
    bg = Image.new("P", (w, h), color=bg_idx)
    ensure_palette(bg, palette)
    bg.paste(res, ((w - nw) // 2, (h - nh) // 2))
    return bg

def shear_P(imgP: Image.Image, shx: float, shy: float, bg_idx: int, palette: List[int]) -> Image.Image:
    # PIL uses output->input mapping
    a, b, c = 1.0, -shx, 0.0
    d, e, f = -shy, 1.0, 0.0
    out = imgP.transform(imgP.size, Image.AFFINE, (a, b, c, d, e, f),
                         resample=Image.NEAREST, fillcolor=bg_idx)
    ensure_palette(out, palette)
    return out

def translate_P(imgP: Image.Image, dx: int, dy: int, bg_idx: int, palette: List[int]) -> Image.Image:
    a, b, c = 1.0, 0.0, -dx
    d, e, f = 0.0, 1.0, -dy
    out = imgP.transform(imgP.size, Image.AFFINE, (a, b, c, d, e, f),
                         resample=Image.NEAREST, fillcolor=bg_idx)
    ensure_palette(out, palette)
    return out

def random_augment_P(imgP: Image.Image, rng: random.Random) -> Image.Image:
    palette = imgP.getpalette()
    bg_idx = guess_background_index(imgP)
    im = imgP.copy()

    if H_FLIP and rng.random() < 0.5:
        im = ImageOps.mirror(im)
        ensure_palette(im, palette)

    z = rng.uniform(1.0 - ZOOM_FRAC, 1.0 + ZOOM_FRAC)
    im = center_zoom_P(im, z, bg_idx, palette)

    shx = rng.uniform(-SHEAR_X_MAX, SHEAR_X_MAX)
    shy = rng.uniform(-SHEAR_Y_MAX, SHEAR_Y_MAX)
    im = shear_P(im, shx, shy, bg_idx, palette)

    angle = rng.uniform(-ROTATION_DEG, ROTATION_DEG)
    im = im.rotate(angle, resample=Image.NEAREST, expand=False, fillcolor=bg_idx)
    ensure_palette(im, palette)

    w, h = im.size
    dx = int(round(rng.uniform(-SHIFT_FRAC, SHIFT_FRAC) * w))
    dy = int(round(rng.uniform(-SHIFT_FRAC, SHIFT_FRAC) * h))
    im = translate_P(im, dx, dy, bg_idx, palette)

    return im

def upscale_P(imgP: Image.Image) -> Image.Image:
    scale = compute_scale(imgP.width, imgP.height)
    if scale > 1:
        palette = imgP.getpalette()
        imgP = imgP.resize((imgP.width * scale, imgP.height * scale), resample=Image.NEAREST)
        ensure_palette(imgP, palette)
    return imgP

# ------------------------------ Main ------------------------------

def main() -> None:
    rng = random.Random(SEED)
    files = [f for f in sorted(os.listdir(original_dir)) if f.lower().endswith(('.png', '.gif'))]
    if not files:
        print(f"No images found in {original_dir}")
        return

    for i, fname in enumerate(files):
        src = os.path.join(original_dir, fname)
        base = os.path.splitext(fname)[0]

        # Keep palette; if input isn't 'P', quantize once to a palette
        im = Image.open(src)
        if im.mode != "P":
            im = im.convert("P")

        for j in range(1, AUGS_PER_IMAGE + 1):
            aug = random_augment_P(im, rng)
            aug = upscale_P(aug)  # no-op unless TARGET_LONG_SIDE_PX set
            out_path = os.path.join(augmented_dir, f"{base}_aug{j:02d}.png")
            aug.save(out_path, format="PNG", optimize=True, compress_level=9)
            print(f"Saved augmented facies image: {out_path} ({aug.width}x{aug.height}px)")

if __name__ == "__main__":
    main()


# 3. Permeability Augmentation

In [ ]:
# /home/g202103050/Documents/A.Rahman/2-3D GAN part/2D/2D_New Eara/augment_perm_pal_only_nn.py
from __future__ import annotations

import os
import math
import random
from typing import List, Tuple

import numpy as np
from PIL import Image, ImageOps

# ----------------------------- Paths -----------------------------
original_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_Perm'
augmented_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Perm_augmented'
os.makedirs(augmented_dir, exist_ok=True)

# ------------------------- Augment Config -------------------------
AUGS_PER_IMAGE = 30
SEED = 2025

ROTATION_DEG = 10          # ±deg
SHIFT_FRAC   = 0.10        # fraction of width/height
SHEAR_X_MAX  = 0.10        # tan(theta) along x
SHEAR_Y_MAX  = 0.10        # tan(theta) along y
ZOOM_FRAC    = 0.10        # ±10%
H_FLIP       = True

# Optional upscale (leave None to keep original px size)
TARGET_LONG_SIDE_PX = None
MIN_SCALE = 1

# ----------------------------- Helpers ----------------------------

def compute_scale(w: int, h: int) -> int:
    need = (TARGET_LONG_SIDE_PX / max(w, h)) if TARGET_LONG_SIDE_PX else 1.0
    return max(MIN_SCALE, int(math.ceil(need)))

def guess_background_index(imP: Image.Image) -> int:
    """Pick modal palette index on 2-px border as background (NODATA)."""
    idx = np.array(imP, dtype=np.uint8)
    border = np.concatenate(
        [idx[:2, :].ravel(), idx[-2:, :].ravel(),
         idx[:, :2].ravel(), idx[:, -2:].ravel()]
    )
    vals, counts = np.unique(border, return_counts=True)
    return int(vals[np.argmax(counts)])

def ensure_palette(imgP: Image.Image, palette: List[int]) -> None:
    """Re-apply original palette after ops that may reset it."""
    if palette:
        imgP.putpalette(palette)

def center_zoom_P(imgP: Image.Image, scale: float, bg_idx: int, palette: List[int]) -> Image.Image:
    w, h = imgP.size
    nw, nh = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    res = imgP.resize((nw, nh), resample=Image.NEAREST)
    ensure_palette(res, palette)
    if scale >= 1.0:
        left, top = (nw - w) // 2, (nh - h) // 2
        out = res.crop((left, top, left + w, top + h))
        ensure_palette(out, palette)
        return out
    bg = Image.new("P", (w, h), color=bg_idx)
    ensure_palette(bg, palette)
    bg.paste(res, ((w - nw) // 2, (h - nh) // 2))
    return bg

def shear_P(imgP: Image.Image, shx: float, shy: float, bg_idx: int, palette: List[int]) -> Image.Image:
    a, b, c = 1.0, -shx, 0.0
    d, e, f = -shy, 1.0, 0.0
    out = imgP.transform(
        imgP.size,
        Image.AFFINE,
        (a, b, c, d, e, f),
        resample=Image.NEAREST,
        fillcolor=bg_idx,
    )
    ensure_palette(out, palette)
    return out

def translate_P(imgP: Image.Image, dx: int, dy: int, bg_idx: int, palette: List[int]) -> Image.Image:
    a, b, c = 1.0, 0.0, -dx
    d, e, f = 0.0, 1.0, -dy
    out = imgP.transform(
        imgP.size,
        Image.AFFINE,
        (a, b, c, d, e, f),
        resample=Image.NEAREST,
        fillcolor=bg_idx,
    )
    ensure_palette(out, palette)
    return out

def random_augment_P(imgP: Image.Image, rng: random.Random) -> Image.Image:
    """Augment a paletted permeability image with NN-safe transforms."""
    palette = imgP.getpalette()
    bg_idx = guess_background_index(imgP)
    im = imgP.copy()

    if H_FLIP and rng.random() < 0.5:
        im = ImageOps.mirror(im)
        ensure_palette(im, palette)

    z = rng.uniform(1.0 - ZOOM_FRAC, 1.0 + ZOOM_FRAC)
    im = center_zoom_P(im, z, bg_idx, palette)

    shx = rng.uniform(-SHEAR_X_MAX, SHEAR_X_MAX)
    shy = rng.uniform(-SHEAR_Y_MAX, SHEAR_Y_MAX)
    im = shear_P(im, shx, shy, bg_idx, palette)

    angle = rng.uniform(-ROTATION_DEG, ROTATION_DEG)
    im = im.rotate(angle, resample=Image.NEAREST, expand=False, fillcolor=bg_idx)
    ensure_palette(im, palette)

    w, h = im.size
    dx = int(round(rng.uniform(-SHIFT_FRAC, SHIFT_FRAC) * w))
    dy = int(round(rng.uniform(-SHIFT_FRAC, SHIFT_FRAC) * h))
    im = translate_P(im, dx, dy, bg_idx, palette)

    return im

def upscale_P(imgP: Image.Image) -> Image.Image:
    scale = compute_scale(imgP.width, imgP.height)
    if scale > 1:
        palette = imgP.getpalette()
        imgP = imgP.resize(
            (imgP.width * scale, imgP.height * scale),
            resample=Image.NEAREST,
        )
        ensure_palette(imgP, palette)
    return imgP

# ------------------------------ Main ------------------------------

def main() -> None:
    rng = random.Random(SEED)
    files = [
        f for f in sorted(os.listdir(original_dir))
        if f.lower().endswith((".png", ".gif"))
    ]
    if not files:
        print(f"No images found in {original_dir}")
        return

    for fname in files:
        src = os.path.join(original_dir, fname)
        base = os.path.splitext(fname)[0]

        im = Image.open(src)
        # Ensure palette mode – your exported perm layers are already "P"
        if im.mode != "P":
            im = im.convert("P")

        for j in range(1, AUGS_PER_IMAGE + 1):
            aug = random_augment_P(im, rng)
            aug = upscale_P(aug)
            out_path = os.path.join(augmented_dir, f"{base}_aug{j:02d}.png")
            aug.save(out_path, format="PNG", optimize=True, compress_level=9)
            print(
                f"Saved augmented permeability image: {out_path} "
                f"({aug.width}x{aug.height}px)"
            )

if __name__ == "__main__":
    main()


# 4. Vshale Augmentation

In [ ]:
# /home/g202103050/Documents/A.Rahman/2-3D GAN part/2D/2D_New Eara/augment_facies_pal_only_nn.py
from __future__ import annotations

import os
import math
import random
from typing import List, Tuple

import numpy as np
from PIL import Image, ImageOps

# ----------------------------- Paths -----------------------------
original_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_VShale'
augmented_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/VShale_augmented'
os.makedirs(augmented_dir, exist_ok=True)

# ------------------------- Augment Config -------------------------
AUGS_PER_IMAGE = 30
SEED = 2025

ROTATION_DEG = 10          # ±deg
SHIFT_FRAC   = 0.10        # fraction of width/height
SHEAR_X_MAX  = 0.10        # tan(theta) along x
SHEAR_Y_MAX  = 0.10        # tan(theta) along y
ZOOM_FRAC    = 0.10        # ±10%
H_FLIP       = True

# Optional upscale (leave None to keep original px size)
TARGET_LONG_SIDE_PX = None
MIN_SCALE = 1

# ----------------------------- Helpers ----------------------------

def compute_scale(w: int, h: int) -> int:
    need = (TARGET_LONG_SIDE_PX / max(w, h)) if TARGET_LONG_SIDE_PX else 1.0
    return max(MIN_SCALE, int(math.ceil(need)))

def guess_background_index(imP: Image.Image) -> int:
    """Use the modal value on a 2px border as background (keeps 'no data' color)."""
    idx = np.array(imP, dtype=np.uint8)
    border = np.concatenate([idx[:2, :].ravel(), idx[-2:, :].ravel(),
                             idx[:, :2].ravel(), idx[:, -2:].ravel()])
    vals, counts = np.unique(border, return_counts=True)
    return int(vals[np.argmax(counts)])

def ensure_palette(imgP: Image.Image, palette: List[int]) -> None:
    """Why: some ops can drop/alter palette; force the original back."""
    if palette:
        imgP.putpalette(palette)

def center_zoom_P(imgP: Image.Image, scale: float, bg_idx: int, palette: List[int]) -> Image.Image:
    w, h = imgP.size
    nw, nh = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    res = imgP.resize((nw, nh), resample=Image.NEAREST)
    ensure_palette(res, palette)
    if scale >= 1.0:
        left, top = (nw - w) // 2, (nh - h) // 2
        out = res.crop((left, top, left + w, top + h))
        ensure_palette(out, palette)
        return out
    bg = Image.new("P", (w, h), color=bg_idx)
    ensure_palette(bg, palette)
    bg.paste(res, ((w - nw) // 2, (h - nh) // 2))
    return bg

def shear_P(imgP: Image.Image, shx: float, shy: float, bg_idx: int, palette: List[int]) -> Image.Image:
    # PIL uses output->input mapping
    a, b, c = 1.0, -shx, 0.0
    d, e, f = -shy, 1.0, 0.0
    out = imgP.transform(imgP.size, Image.AFFINE, (a, b, c, d, e, f),
                         resample=Image.NEAREST, fillcolor=bg_idx)
    ensure_palette(out, palette)
    return out

def translate_P(imgP: Image.Image, dx: int, dy: int, bg_idx: int, palette: List[int]) -> Image.Image:
    a, b, c = 1.0, 0.0, -dx
    d, e, f = 0.0, 1.0, -dy
    out = imgP.transform(imgP.size, Image.AFFINE, (a, b, c, d, e, f),
                         resample=Image.NEAREST, fillcolor=bg_idx)
    ensure_palette(out, palette)
    return out

def random_augment_P(imgP: Image.Image, rng: random.Random) -> Image.Image:
    palette = imgP.getpalette()
    bg_idx = guess_background_index(imgP)
    im = imgP.copy()

    if H_FLIP and rng.random() < 0.5:
        im = ImageOps.mirror(im)
        ensure_palette(im, palette)

    z = rng.uniform(1.0 - ZOOM_FRAC, 1.0 + ZOOM_FRAC)
    im = center_zoom_P(im, z, bg_idx, palette)

    shx = rng.uniform(-SHEAR_X_MAX, SHEAR_X_MAX)
    shy = rng.uniform(-SHEAR_Y_MAX, SHEAR_Y_MAX)
    im = shear_P(im, shx, shy, bg_idx, palette)

    angle = rng.uniform(-ROTATION_DEG, ROTATION_DEG)
    im = im.rotate(angle, resample=Image.NEAREST, expand=False, fillcolor=bg_idx)
    ensure_palette(im, palette)

    w, h = im.size
    dx = int(round(rng.uniform(-SHIFT_FRAC, SHIFT_FRAC) * w))
    dy = int(round(rng.uniform(-SHIFT_FRAC, SHIFT_FRAC) * h))
    im = translate_P(im, dx, dy, bg_idx, palette)

    return im

def upscale_P(imgP: Image.Image) -> Image.Image:
    scale = compute_scale(imgP.width, imgP.height)
    if scale > 1:
        palette = imgP.getpalette()
        imgP = imgP.resize((imgP.width * scale, imgP.height * scale), resample=Image.NEAREST)
        ensure_palette(imgP, palette)
    return imgP

# ------------------------------ Main ------------------------------

def main() -> None:
    rng = random.Random(SEED)
    files = [f for f in sorted(os.listdir(original_dir)) if f.lower().endswith(('.png', '.gif'))]
    if not files:
        print(f"No images found in {original_dir}")
        return

    for i, fname in enumerate(files):
        src = os.path.join(original_dir, fname)
        base = os.path.splitext(fname)[0]

        # Keep palette; if input isn't 'P', quantize once to a palette
        im = Image.open(src)
        if im.mode != "P":
            im = im.convert("P")

        for j in range(1, AUGS_PER_IMAGE + 1):
            aug = random_augment_P(im, rng)
            aug = upscale_P(aug)  # no-op unless TARGET_LONG_SIDE_PX set
            out_path = os.path.join(augmented_dir, f"{base}_aug{j:02d}.png")
            aug.save(out_path, format="PNG", optimize=True, compress_level=9)
            print(f"Saved augmented facies image: {out_path} ({aug.width}x{aug.height}px)")

if __name__ == "__main__":
    main()
